In [ ]:
"""
Building a Relational Paper Mill & Production Run Quality Dashboard
Roll No: 25SSDS415085  |  Name: Vipraja

Single-file Python application (tkinter + sqlite3).

Tables
    mills            (mill_id, mill_name UNIQUE, contact_email)
    production_runs  (run_id, mill_id -> mills, paper_grade, run_date)
    quality_items    (item_id, run_id -> production_runs, quality_metric, measured_value)

Rules
    * Foreign keys are switched ON for every connection.
    * ON DELETE CASCADE: deleting a mill removes its runs and their quality items;
      deleting a run removes its quality items; deleting a quality item removes only that row.
    * Database is seeded with sample data when it is completely empty.

How to run
    python paper_mill_dashboard.py            -> opens the GUI (VS Code / local Python)
    python paper_mill_dashboard.py --selftest -> runs database checks in the console (no GUI)
    Jupyter Notebook (local)                  -> paste the whole file into one cell and run it

If no graphical display is available (for example Google Colab or a remote server),
the script prints a clear message and runs the console self-test instead of crashing.
"""

import math
import os
import sqlite3
import sys
from datetime import date

try:  # normal script (VS Code / terminal)
    _BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:  # Jupyter notebook: __file__ does not exist, use the working folder
    _BASE_DIR = os.getcwd()
DB_FILE = os.path.join(_BASE_DIR, "paper_mill.db")

SUGGESTED_METRICS = ["Pulp Consistency %", "Brightness Index", "GSM Weight"]


# ----------------------------------------------------------------------------
# DATABASE LAYER 
# ----------------------------------------------------------------------------
def connect(db_file=DB_FILE):
    """Open a connection with foreign key enforcement switched on."""
    conn = sqlite3.connect(db_file)
    conn.execute("PRAGMA foreign_keys = ON")
    return conn


def create_schema(conn):
    conn.executescript(
        """
        CREATE TABLE IF NOT EXISTS mills (
            mill_id       INTEGER PRIMARY KEY AUTOINCREMENT,
            mill_name     TEXT NOT NULL UNIQUE COLLATE NOCASE,
            contact_email TEXT NOT NULL
        );
        CREATE TABLE IF NOT EXISTS production_runs (
            run_id      INTEGER PRIMARY KEY AUTOINCREMENT,
            mill_id     INTEGER NOT NULL,
            paper_grade TEXT NOT NULL,
            run_date    TEXT NOT NULL,
            FOREIGN KEY (mill_id) REFERENCES mills(mill_id) ON DELETE CASCADE
        );
        CREATE TABLE IF NOT EXISTS quality_items (
            item_id        INTEGER PRIMARY KEY AUTOINCREMENT,
            run_id         INTEGER NOT NULL,
            quality_metric TEXT NOT NULL,
            measured_value REAL NOT NULL CHECK (measured_value >= 0),
            FOREIGN KEY (run_id) REFERENCES production_runs(run_id) ON DELETE CASCADE
        );
        """
    )
    conn.commit()


def seed_if_empty(conn):
    """Insert sample rows only if the database has no mills at all."""
    if conn.execute("SELECT COUNT(*) FROM mills").fetchone()[0] > 0:
        return
    mills = [
        ("Himalaya Paper Mills", "ops@himalayapaper.example"),
        ("Coastal Pulp & Paper", "quality@coastalpulp.example"),
        ("Deccan Board Works", "plant@deccanboard.example"),
    ]
    conn.executemany("INSERT INTO mills (mill_name, contact_email) VALUES (?, ?)", mills)
    runs = [
        (1, "A4 Copier 80 GSM", "2026-09-02"),
        (1, "Kraft Liner Board", "2026-09-14"),
        (2, "Newsprint 45 GSM", "2026-09-20"),
        (2, "A4 Copier 80 GSM", "2026-10-03"),
        (3, "Duplex Board 300 GSM", "2026-10-11"),
    ]
    conn.executemany(
        "INSERT INTO production_runs (mill_id, paper_grade, run_date) VALUES (?, ?, ?)", runs
    )
    items = [
        (1, "Pulp Consistency %", 3.4),
        (1, "Brightness Index", 92.5),
        (1, "GSM Weight", 80.1),
        (2, "GSM Weight", 150.0),
        (3, "Brightness Index", 58.2),
        (3, "Pulp Consistency %", 2.9),
        (4, "GSM Weight", 79.8),
        (5, "GSM Weight", 301.2),
        (5, "Brightness Index", 76.0),
    ]
    conn.executemany(
        "INSERT INTO quality_items (run_id, quality_metric, measured_value) VALUES (?, ?, ?)",
        items,
    )
    conn.commit()


def init_database(db_file=DB_FILE):
    conn = connect(db_file)
    create_schema(conn)
    seed_if_empty(conn)
    return conn


class ValidationError(ValueError):
    """Raised when user input fails validation (message is shown to the user)."""


def validate_inputs(mill_name, paper_grade, quality_metric, value_text):
    """Return cleaned (mill, grade, metric, value). Raise ValidationError if invalid."""
    mill_name = (mill_name or "").strip()
    paper_grade = (paper_grade or "").strip()
    quality_metric = (quality_metric or "").strip()
    value_text = (value_text or "").strip()

    missing = []
    if not mill_name:
        missing.append("Mill Name")
    if not paper_grade:
        missing.append("Paper Grade")
    if not quality_metric:
        missing.append("Quality Metric")
    if not value_text:
        missing.append("Measured Value")
    if missing:
        raise ValidationError("Please fill in: " + ", ".join(missing) + ".")

    try:
        value = float(value_text)
    except ValueError:
        raise ValidationError("Measured Value must be a number (for example 80.5).")
    if not math.isfinite(value):
        raise ValidationError("Measured Value must be a real number.")
    if value < 0:
        raise ValidationError("Measured Value cannot be negative.")
    return mill_name, paper_grade, quality_metric, value


def add_record(conn, mill_name, paper_grade, quality_metric, value_text):
    """
    Add one production run with one quality item.
    If the mill already exists (case-insensitive) its id is reused,
    otherwise a new mill profile is created first.
    Returns (mill_id, run_id, item_id, created_new_mill).
    """
    mill_name, paper_grade, quality_metric, value = validate_inputs(
        mill_name, paper_grade, quality_metric, value_text
    )
    with conn:  # one transaction: all three inserts succeed or none do
        row = conn.execute(
            "SELECT mill_id FROM mills WHERE mill_name = ?", (mill_name,)
        ).fetchone()
        created = row is None
        if created:
            email = "".join(c.lower() if c.isalnum() else "." for c in mill_name)
            email = ".".join(part for part in email.split(".") if part) + "@papermill.example"
            cur = conn.execute(
                "INSERT INTO mills (mill_name, contact_email) VALUES (?, ?)", (mill_name, email)
            )
            mill_id = cur.lastrowid
        else:
            mill_id = row[0]
        cur = conn.execute(
            "INSERT INTO production_runs (mill_id, paper_grade, run_date) VALUES (?, ?, ?)",
            (mill_id, paper_grade, date.today().isoformat()),
        )
        run_id = cur.lastrowid
        cur = conn.execute(
            "INSERT INTO quality_items (run_id, quality_metric, measured_value) VALUES (?, ?, ?)",
            (run_id, quality_metric, value),
        )
        item_id = cur.lastrowid
    return mill_id, run_id, item_id, created


# Delete targets: label -> (table, id column)
DELETE_TARGETS = {
    "Paper Mill": ("mills", "mill_id"),
    "Production Run": ("production_runs", "run_id"),
    "Quality Item": ("quality_items", "item_id"),
}


def list_delete_options(conn, target):
    """Return [(label, id), ...] for the chosen delete target, read live from the database."""
    if target == "Paper Mill":
        rows = conn.execute("SELECT mill_id, mill_name FROM mills ORDER BY mill_id").fetchall()
        return [(f"Mill {i}: {name}", i) for i, name in rows]
    if target == "Production Run":
        rows = conn.execute(
            """SELECT r.run_id, m.mill_name, r.paper_grade, r.run_date
               FROM production_runs r JOIN mills m ON m.mill_id = r.mill_id
               ORDER BY r.run_id"""
        ).fetchall()
        return [(f"Run {i}: {m} / {g} / {d}", i) for i, m, g, d in rows]
    if target == "Quality Item":
        rows = conn.execute(
            """SELECT q.item_id, q.run_id, q.quality_metric, q.measured_value
               FROM quality_items q ORDER BY q.item_id"""
        ).fetchall()
        return [(f"Item {i} (Run {r}): {m} = {v:g}", i) for i, r, m, v in rows]
    return []


def delete_record(conn, target, record_id):
    """Delete one row; ON DELETE CASCADE removes child rows. Returns rows deleted from the target table."""
    table, id_col = DELETE_TARGETS[target]
    with conn:
        cur = conn.execute(f"DELETE FROM {table} WHERE {id_col} = ?", (record_id,))
    return cur.rowcount


# Queries for the four tabs: (tab title, column headings, SQL)
VIEWS = [
    (
        "Master Quality View (JOIN)",
        ("Run ID", "Mill Name", "Paper Grade", "Run Date", "Quality Metric", "Measured Value"),
        """SELECT r.run_id, m.mill_name, r.paper_grade, r.run_date,
                  q.quality_metric, q.measured_value
           FROM production_runs r
           JOIN mills m ON m.mill_id = r.mill_id
           LEFT JOIN quality_items q ON q.run_id = r.run_id
           ORDER BY r.run_id, q.item_id""",
    ),
    (
        "Mills",
        ("mill_id", "mill_name", "contact_email"),
        "SELECT mill_id, mill_name, contact_email FROM mills ORDER BY mill_id",
    ),
    (
        "Production Runs",
        ("run_id", "mill_id", "paper_grade", "run_date"),
        "SELECT run_id, mill_id, paper_grade, run_date FROM production_runs ORDER BY run_id",
    ),
    (
        "Quality Items",
        ("item_id", "run_id", "quality_metric", "measured_value"),
        "SELECT item_id, run_id, quality_metric, measured_value FROM quality_items ORDER BY item_id",
    ),
]


# ----------------------------------------------------------------------------
# CONSOLE SELF-TEST 
# ----------------------------------------------------------------------------
def selftest():
    import tempfile

    path = os.path.join(tempfile.mkdtemp(), "test.db")
    conn = init_database(path)

    def count(table):
        return conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]

    print("Seeded:", count("mills"), "mills,", count("production_runs"), "runs,",
          count("quality_items"), "items")
    assert (count("mills"), count("production_runs"), count("quality_items")) == (3, 5, 9)

    # seeding must not repeat on a non-empty database
    seed_if_empty(conn)
    assert count("mills") == 3

    # validation
    for bad in [("", "A4", "GSM", "5"), ("M", "", "GSM", "5"), ("M", "A4", "", "5"),
                ("M", "A4", "GSM", ""), ("M", "A4", "GSM", "abc"), ("M", "A4", "GSM", "-1"),
                ("M", "A4", "GSM", "nan"), ("M", "A4", "GSM", "inf")]:
        try:
            add_record(conn, *bad)
            raise SystemExit(f"validation failed to reject {bad}")
        except ValidationError:
            pass
    assert count("mills") == 3 and count("production_runs") == 5

    # existing mill reused (case-insensitive), new mill created
    m, r, i, created = add_record(conn, "himalaya paper mills", "Tissue 20 GSM", "GSM Weight", "20.2")
    assert m == 1 and created is False and count("mills") == 3
    m, r, i, created = add_record(conn, "Sahyadri Papers", "Kraft", "Brightness Index", "55")
    assert created is True and count("mills") == 4

    # delete a quality item only removes that row
    before = (count("mills"), count("production_runs"))
    assert delete_record(conn, "Quality Item", i) == 1
    assert (count("mills"), count("production_runs")) == before

    # delete a run cascades to its items
    run_items = conn.execute("SELECT COUNT(*) FROM quality_items WHERE run_id = 1").fetchone()[0]
    assert run_items == 3
    delete_record(conn, "Production Run", 1)
    assert conn.execute("SELECT COUNT(*) FROM quality_items WHERE run_id = 1").fetchone()[0] == 0

    # delete a mill cascades to runs and items
    delete_record(conn, "Paper Mill", 2)
    assert conn.execute("SELECT COUNT(*) FROM production_runs WHERE mill_id = 2").fetchone()[0] == 0
    orphans = conn.execute(
        "SELECT COUNT(*) FROM quality_items WHERE run_id NOT IN (SELECT run_id FROM production_runs)"
    ).fetchone()[0]
    assert orphans == 0

    # dropdown lists never contain deleted ids
    assert 2 not in [i for _, i in list_delete_options(conn, "Paper Mill")]
    assert 1 not in [i for _, i in list_delete_options(conn, "Production Run")]

    # foreign key enforcement is on
    try:
        conn.execute("INSERT INTO production_runs (mill_id, paper_grade, run_date) VALUES (999, 'x', 'y')")
        raise SystemExit("foreign key not enforced")
    except sqlite3.IntegrityError:
        pass
    conn.close()
    print("All database self-tests passed.")


# ----------------------------------------------------------------------------
# GUI LAYER
# ----------------------------------------------------------------------------
def run_gui():
    import tkinter as tk
    from tkinter import messagebox, ttk

    try:
        root = tk.Tk()
    except tk.TclError as err:
        print("No graphical display is available (" + str(err) + ").")
        print("This happens on Google Colab, remote servers and some notebooks.")
        print("Run this file in VS Code or a local terminal to see the window.")
        print("Running the console self-test instead:\n")
        selftest()
        return

    conn = init_database()

    class App:
        def __init__(self, master):
            self.root = master
            self.root.title("Paper Mill & Production Run Quality Dashboard")
            self.root.geometry("1020x720")
            self.root.minsize(900, 640)

            self.mill_var = tk.StringVar()
            self.grade_var = tk.StringVar()
            self.metric_var = tk.StringVar()
            self.value_var = tk.StringVar()
            self.target_var = tk.StringVar(value="Paper Mill")
            self.choice_var = tk.StringVar()
            self.choice_map = {}
            self.trees = []

            ttk.Label(
                self.root, text="Paper Mill & Production Run Quality Dashboard",
                font=("Helvetica", 16, "bold"),
            ).pack(pady=(10, 4))

            top = ttk.Frame(self.root)
            top.pack(fill="x", padx=10, pady=4)
            top.columnconfigure(0, weight=1)
            top.columnconfigure(1, weight=1)
            self.build_insert_panel(top)
            self.build_delete_panel(top)
            self.build_tabs()

            self.status_var = tk.StringVar(value="Ready.")
            ttk.Label(self.root, textvariable=self.status_var, anchor="w",
                      relief="sunken").pack(fill="x", side="bottom")
            self.refresh_all()

        # ---------------- Insert panel
        def build_insert_panel(self, parent):
            box = ttk.LabelFrame(parent, text="Insert Panel")
            box.grid(row=0, column=0, sticky="nsew", padx=(0, 5))
            box.columnconfigure(1, weight=1)
            rows = [("Mill Name", self.mill_var), ("Paper Grade", self.grade_var)]
            for r, (text, var) in enumerate(rows):
                ttk.Label(box, text=text).grid(row=r, column=0, sticky="w", padx=6, pady=4)
                ttk.Entry(box, textvariable=var, width=32).grid(
                    row=r, column=1, sticky="ew", padx=6, pady=4)
            ttk.Label(box, text="Quality Metric").grid(row=2, column=0, sticky="w", padx=6, pady=4)
            ttk.Combobox(box, textvariable=self.metric_var, values=SUGGESTED_METRICS,
                         width=30).grid(row=2, column=1, sticky="ew", padx=6, pady=4)
            ttk.Label(box, text="Measured Value").grid(row=3, column=0, sticky="w", padx=6, pady=4)
            ttk.Entry(box, textvariable=self.value_var, width=32).grid(
                row=3, column=1, sticky="ew", padx=6, pady=4)
            ttk.Button(box, text="Add Record", command=self.on_add).grid(
                row=4, column=0, columnspan=2, pady=8)

        # ---------------- Delete panel
        def build_delete_panel(self, parent):
            box = ttk.LabelFrame(parent, text="Delete Panel")
            box.grid(row=0, column=1, sticky="nsew", padx=(5, 0))
            box.columnconfigure(1, weight=1)
            ttk.Label(box, text="Delete a").grid(row=0, column=0, sticky="w", padx=6, pady=4)
            target = ttk.Combobox(box, textvariable=self.target_var, state="readonly",
                                  values=list(DELETE_TARGETS.keys()), width=28)
            target.grid(row=0, column=1, sticky="ew", padx=6, pady=4)
            target.bind("<<ComboboxSelected>>", lambda e: self.refresh_delete_choices())
            ttk.Label(box, text="Select record").grid(row=1, column=0, sticky="w", padx=6, pady=4)
            self.choice_box = ttk.Combobox(box, textvariable=self.choice_var, state="readonly",
                                           width=45)
            self.choice_box.grid(row=1, column=1, sticky="ew", padx=6, pady=4)
            ttk.Button(box, text="Delete Selected", command=self.on_delete).grid(
                row=2, column=0, columnspan=2, pady=8)
            ttk.Label(box, wraplength=380, foreground="#555555",
                      text="Deleting a mill also removes its runs and quality items. "
                           "Deleting a run removes its quality items. "
                           "Deleting a quality item removes only that row.").grid(
                row=3, column=0, columnspan=2, padx=6, pady=(0, 6))

        # ---------------- Tabbed display
        def build_tabs(self):
            self.notebook = ttk.Notebook(self.root)
            self.notebook.pack(fill="both", expand=True, padx=10, pady=6)
            for title, columns, _sql in VIEWS:
                frame = ttk.Frame(self.notebook)
                self.notebook.add(frame, text=title)
                tree = ttk.Treeview(frame, columns=columns, show="headings")
                for col in columns:
                    tree.heading(col, text=col)
                    tree.column(col, width=130, anchor="w")
                ys = ttk.Scrollbar(frame, orient="vertical", command=tree.yview)
                xs = ttk.Scrollbar(frame, orient="horizontal", command=tree.xview)
                tree.configure(yscrollcommand=ys.set, xscrollcommand=xs.set)
                tree.grid(row=0, column=0, sticky="nsew")
                ys.grid(row=0, column=1, sticky="ns")
                xs.grid(row=1, column=0, sticky="ew")
                frame.rowconfigure(0, weight=1)
                frame.columnconfigure(0, weight=1)
                self.trees.append(tree)

        # ---------------- Refresh helpers
        def refresh_tabs(self):
            for tree, (_title, _cols, sql) in zip(self.trees, VIEWS):
                tree.delete(*tree.get_children())
                for row in conn.execute(sql).fetchall():
                    tree.insert("", "end", values=row)

        def refresh_delete_choices(self):
            """Reload the record dropdown from the database so it never shows a deleted id."""
            options = list_delete_options(conn, self.target_var.get())
            self.choice_map = {label: rec_id for label, rec_id in options}
            self.choice_box["values"] = [label for label, _ in options]
            self.choice_var.set("")

        def refresh_all(self):
            self.refresh_tabs()
            self.refresh_delete_choices()

        def clear_inputs(self):
            for var in (self.mill_var, self.grade_var, self.metric_var, self.value_var):
                var.set("")

        # ---------------- Button handlers
        def on_add(self):
            try:
                mill_id, run_id, item_id, created = add_record(
                    conn, self.mill_var.get(), self.grade_var.get(),
                    self.metric_var.get(), self.value_var.get())
            except ValidationError as err:
                messagebox.showerror("Invalid input", str(err))
                return
            except sqlite3.Error as err:
                messagebox.showerror("Database error", str(err))
                return
            note = "new mill created" if created else "existing mill reused"
            self.status_var.set(
                f"Added run {run_id} with quality item {item_id} for mill {mill_id} ({note}).")
            self.clear_inputs()
            self.refresh_all()

        def on_delete(self):
            label = self.choice_var.get()
            if not label or label not in self.choice_map:
                messagebox.showwarning("Nothing selected", "Please select a record to delete.")
                return
            target = self.target_var.get()
            if not messagebox.askyesno("Confirm delete", f"Delete {label}?\nThis cannot be undone."):
                return
            try:
                deleted = delete_record(conn, target, self.choice_map[label])
            except sqlite3.Error as err:
                messagebox.showerror("Database error", str(err))
                return
            self.status_var.set(f"Deleted {label}." if deleted else "Record was already gone.")
            self.refresh_all()

    App(root)
    # bring the window to the front 
    root.lift()
    root.attributes("-topmost", True)
    root.after(400, lambda: root.attributes("-topmost", False))
    root.mainloop()
    conn.close()


if __name__ == "__main__":
    if "--selftest" in sys.argv:
        selftest()
    else:
        run_gui()